# 10 可观测日志、Harness与ModelRouter

**用途：** 检查模型路由、错误分类、脱敏、预算与本地可观测性。

> 使用方式：按顺序运行。出现 `PASS` 才代表本节验收成功；断言失败时先阅读紧邻的“失败定位”。默认不调用真实模型、不写生产数据库。

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys
import tempfile

cwd = Path.cwd().resolve()
DAY1_ROOT = None
PROJECT2_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "project2" / "agent_graph.py").exists():
        DAY1_ROOT = candidate
        PROJECT2_ROOT = candidate / "project2"
        break
    if (candidate / "agent_graph.py").exists() and (candidate / "tests").exists():
        PROJECT2_ROOT = candidate
        DAY1_ROOT = candidate.parent
        break
assert DAY1_ROOT is not None and PROJECT2_ROOT is not None, "找不到 day1/project2 项目根目录"
NOTEBOOK_ROOT = PROJECT2_ROOT / "notebooks"
for path in [str(DAY1_ROOT), str(PROJECT2_ROOT), str(NOTEBOOK_ROOT)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from notebook_utils import (
    check,
    check_equal,
    file_inventory,
    load_jsonl,
    masked_environment,
    run_command,
    run_unittest,
    show_markdown,
    show_table,
    source_excerpt,
)

RUN_LIVE_MODEL_TESTS = os.getenv("RUN_LIVE_MODEL_TESTS", "0") == "1"
print(f"Python: {sys.executable}")
print(f"DAY1_ROOT: {DAY1_ROOT}")
print(f"PROJECT2_ROOT: {PROJECT2_ROOT}")
print(f"RUN_LIVE_MODEL_TESTS: {RUN_LIVE_MODEL_TESTS}")

Python: D:\new things\项目1\day1\.venv\Scripts\python.exe
DAY1_ROOT: D:\new things\项目1\day1
PROJECT2_ROOT: D:\new things\项目1\day1\project2
RUN_LIVE_MODEL_TESTS: False


In [2]:
from model_router import ModelRouter
from agent_harness import classify_model_error, sanitize_error_message

router = ModelRouter.from_env()
description = router.describe()
print(json.dumps(description, ensure_ascii=False, indent=2))
serialized = json.dumps(description, ensure_ascii=False)
check("公开路由只显示密钥配置状态", "api_key_configured" in serialized)
probe_secret = "notebook-probe-secret-value"
probe_router = ModelRouter.from_env({
    "AGENT_TEXT_PROVIDER": "deepseek",
    "DEEPSEEK_API_KEY": probe_secret,
    "AGENT_VISION_PROVIDER": "disabled",
})
probe_serialized = json.dumps(
    probe_router.describe(),
    ensure_ascii=False,
)
check(
    "路由描述不含真实API Key值",
    probe_secret not in probe_serialized,
)

safe_message = sanitize_error_message(
    "Authorization Bearer secret-token-123 api_key=abcdef"
)
print(safe_message)
check("错误日志已脱敏", "secret-token-123" not in safe_message and "abcdef" not in safe_message)

error_type, retryable = classify_model_error(TimeoutError("model timed out"))
show_table([{"错误类型": error_type, "是否重试": retryable}])
check("超时可重试", retryable)

{
  "text": {
    "capability": "text",
    "provider": "deepseek",
    "model": "deepseek-v4-flash",
    "base_url": "https://api.deepseek.com",
    "api_key_env": "DEEPSEEK_API_KEY",
    "api_key_configured": true,
    "timeout_seconds": 30,
    "max_retries": 1,
    "max_output_tokens": 800,
    "input_cost_per_million_cny": 0,
    "output_cost_per_million_cny": 0,
    "configured": true
  },
  "vision": {
    "capability": "vision",
    "provider": "zhipu",
    "model": "glm-4.1v-thinking-flash",
    "base_url": "https://open.bigmodel.cn/api/paas/v4",
    "api_key_env": "ZHIPU_API_KEY",
    "api_key_configured": true,
    "timeout_seconds": 30,
    "max_retries": 1,
    "max_output_tokens": 1200,
    "input_cost_per_million_cny": 0,
    "output_cost_per_million_cny": 0,
    "configured": true
  }
}
[PASS] 公开路由只显示密钥配置状态
[PASS] 路由描述不含真实API Key值
Authorization Bearer [REDACTED] api_key=[REDACTED]
[PASS] 错误日志已脱敏


,错误类型,是否重试
0,timeout,True


[PASS] 超时可重试


{'检查项': '超时可重试', '状态': 'PASS', '说明': ''}

In [3]:
harness_tests = run_unittest(
    ["tests.test_model_harness"],
    project2_root=PROJECT2_ROOT,
)
check("ModelRouter/Harness 12条通过", "Ran 12 tests" in harness_tests.output and "OK" in harness_tests.output)

$ D:\new things\项目1\day1\.venv\Scripts\python.exe -m unittest tests.test_model_harness -v
test_authentication_failure_is_not_retried (tests.test_model_harness.AgentHarnessTests.test_authentication_failure_is_not_retried) ... ok
test_budget_blocks_call_before_operation (tests.test_model_harness.AgentHarnessTests.test_budget_blocks_call_before_operation) ... ok
test_error_message_redacts_api_credentials (tests.test_model_harness.AgentHarnessTests.test_error_message_redacts_api_credentials) ... ok
test_error_message_redacts_short_named_api_key (tests.test_model_harness.AgentHarnessTests.test_error_message_redacts_short_named_api_key) ... ok
test_invalid_structured_response_recovers_once (tests.test_model_harness.AgentHarnessTests.test_invalid_structured_response_recovers_once) ... ok
test_json_decode_error_is_retryable (tests.test_model_harness.AgentHarnessTests.test_json_decode_error_is_retryable) ... ok
test_langchain_parse_exposes_harness_snapshot (tests.test_model_harness.AgentHarness

{'检查项': 'ModelRouter/Harness 12条通过', '状态': 'PASS', '说明': ''}

## Harness解决什么

Harness包住模型调用，统一处理超时、有限重试、错误分类、同步并发、每轮调用/Token/估算费用预算、结构化输出重试和安全telemetry。ModelRouter按`text`和`vision`能力选择Provider，所以引入视觉模型不需要替换DeepSeek文本链。

当前可观测性包括execution trace、工具CSV、模型JSONL、checkpoint历史和离线评测。LangSmith尚未正式接入。

### 面试官会问

1. Harness和普通try/except有什么区别？
2. 哪些错误应该重试，鉴权错误为什么不重试？
3. 预算如何在调用前阻止请求？
4. 日志为什么不能保存原始Prompt、客户电话和图片？
5. LangSmith能补什么，为什么现在不是阻塞项？
6. 分布式限流、熔断和Provider自动降级目前欠缺什么？

### 参考答案

1. **Harness与try/except的区别？** try/except只处理一个调用点；Harness统一路由模型、调用前预算、并发槽位、错误分类、有限重试、结构化响应重试、延迟/Token/费用估算和脱敏telemetry，让文本与视觉调用共享同一治理策略。
2. **哪些错误重试？** 超时、连接中断、限流和偶发无效JSON可以退避后有限重试；鉴权、余额、明确参数错误和业务校验失败重试不会自行恢复，反而增加费用，所以直接失败并进入配置提示或人工兜底。
3. **预算如何调用前阻止？** Harness的ledger记录本轮调用次数和Token估算，在创建模型请求前计算本次预留输出及累计成本；超过`max_calls/max_tokens/max_cost`立即抛出`ModelBudgetExceeded`，因此不会先花费再报警。
4. **为什么不保存原始数据？** Prompt可能含客户电话、订单号和商业信息，图片可能含铭牌、位置和个人信息；完整落日志会扩大泄漏面。当前只保存必要元数据、错误类别、模型、耗时和预算快照，凭据经`sanitize_error_message`处理。
5. **LangSmith能补什么？** 它能提供跨节点Trace、数据集、实验对比、反馈和线上采样观察。当前本地execution trace、JSONL/CSV和离线评测已满足作品集演示，所以LangSmith是增强项，不应为了接平台而替代现有可解释数据。
6. **当前生产缺口是什么？** 现有限流是单进程线程信号量，熔断状态和Provider健康也没有跨实例共享。生产需要Redis/网关限流、滑动窗口熔断、健康探测、主备路由、请求幂等、真实用量回传和告警。

**代码落点：** `agent_harness.py`、`model_router.py`、`execution_trace.py`和`tests/test_model_harness.py`。